# Idea 0 of the ideas planner: oracle test at low altitude
Does LoFTR match at the TRUE place of the low-altitude photos that ours skips? Inject the true position as the only candidate and run LoFTR over heading -15..+15 (step 5) x scale 0.80..1.25.
Reading: best inliers at the truth >= 8 means the pipeline loses these photos upstream (the place is not among its candidates); low inliers at every pose mean LoFTR cannot match them at this resolution / pose, so the fix is in matching or scale / pose, not retrieval.
Controls: low-altitude photos that ours answers right (within 1 sigma), same grid. Our official150 dumps only. Run with GATE_DATA=official.

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

DATA = os.environ.get("GATE_DATA", "official")
MAP = "new"
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))   # same pool and setup seed as Test_Iteration_3, so NUDT seed 101 / 202 pick the same photos
WIDE_HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
WIDE_SCALES = [0.95, 1.0, 1.05]
K_PER_QUERY, M_PER_QUERY = 50, 10
MATCHER_CONFIDENCE, LOFTR_COARSE_THRESHOLD = 0.22, 0.21
MIN_INLIERS, GAMMA = 8, 1.0
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEEDS = [int(s) for s in os.environ.get("EXCLUDE_SEEDS", "").split(",") if s.strip()]
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
EXCLUDE_N = int(os.environ.get("EXCLUDE_N", "300"))   # size of the excluded seeds' samples (seed 101 picked 300; this reproduces Test_Iteration_3's seed 202 photos exactly)
MATCHER = os.environ.get("MATCHER", "loftr")          # "loftr" (the pipeline) or "xoftr" (MINIMA XoFTR, a swap-in with the same plumbing)
XOFTR_COARSE, XOFTR_FINE = float(os.environ.get("XOFTR_COARSE", "0.3")), float(os.environ.get("XOFTR_FINE", "0.1"))
FRESH_EXCLUDE_POOL = int(os.environ.get("FRESH_EXCLUDE_POOL", "0"))   # >0 (NUDT, with POOL_PHOTOS >= all frames): skip every photo of the usual
# pool of this size (setup seed 7), so the sample is from photos no earlier test has used
EXCLUDE_IDS_FILE = os.environ.get("EXCLUDE_IDS_FILE")   # optional csv with a photo column: never draw these photos (an earlier fresh sample)
PHOTOS_FILE = os.environ.get("PHOTOS_FILE")           # optional csv with a photo column: only these photos
print(MATCHER, DATA, "seed", NUDT_SEED, "n", N_NUDT, "exclude", EXCLUDE_SEEDS, "limit", N_LIMIT)

loftr official seed 101 n 300 exclude [] limit 0


In [2]:
from train.checks.rich_dump import RichDumpPipeline, pack_photo
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=GAMMA, k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY,
                            matcher="loftr_minima", min_inliers=MIN_INLIERS, ransac_pixels=None, matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP,
                            map_margin_m=None, loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=None,
                            headings=WIDE_HEADINGS, scale_factors=WIDE_SCALES)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set
if MATCHER == "xoftr":
    from train.checks.xoftr_batched import load_xoftr_batched      # the src loader lacks the batch ids the batched LoFTR plumbing reads
    pipeline.matcher = load_xoftr_batched(ROOT / "dataset/NUDT/checkpoint/MINIMA/minima_xoftr.ckpt", min_confidence=XOFTR_FINE,
                                          coarse_threshold=XOFTR_COARSE, fine_threshold=XOFTR_FINE)
    print("matcher: XoFTR (coarse %.2f, fine %.2f)" % (XOFTR_COARSE, XOFTR_FINE))

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 113 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup done in 129.2 s
data: official | 379 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 27 (headings [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0], scale factors [0.95, 1.0, 1.05]); K and M are per query; 1 queries matched together


In [3]:
import math, gc, torch
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches
from image_processing.geometry import render_query
T7 = ROOT / "tests/Test_Iteration_7/results"
dump = replay.load_set(T7 / "rich_dump" / "official150")
LOW = 150.0
low, control = [], []
for pid, photo in dump.items():
    if photo["height"] > LOW:
        continue
    r = replay.decide_current(photo)
    if r["stage"] == "skip":
        low.append(pid)
    elif r["err_sigma"] <= 1.0:
        control.append(pid)
control = control[:8]
print(len(low), "low-altitude photos ours skips;", len(control), "controls")

12 low-altitude photos ours skips; 8 controls


In [4]:
HEADINGS = [-15.0, -10.0, -5.0, 0.0, 5.0, 10.0, 15.0]
SCALES = [0.80, 0.90, 0.95, 1.0, 1.05, 1.10, 1.25]
rows = []
for group, ids in [("skipped", low), ("control", control)]:
    for pid in ids:
        photo = dump[pid]; item = pipeline.test_set.items[pid]; level = photo["level"]
        tx, ty = [float(v) for v in photo["truth"]]
        peaks = np.array([[1.0, tx, ty, 0.0, 0.0]])
        for h in HEADINGS:
            for s in SCALES:
                canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
                stored = candidate_matches(pipeline.matcher, canvas, valid, level, peaks, pipeline.level_images)
                s2 = stage2_from_matches(stored, level, MATCHER_CONFIDENCE)
                err = float(np.hypot(*s2["shifts"][0])) / photo["sigma"] if s2["inliers"][0] > 0 else np.nan
                rows.append(dict(group=group, photo=pid, height=photo["height"], heading=h, scale=s, inliers=float(s2["inliers"][0]), shift_sigma=err))
        torch.cuda.empty_cache()
O = pd.DataFrame(rows); O.to_csv(T7 / "idea0_oracle.csv", index=False); print(len(O), "runs")

980 runs


In [5]:
summary = []
for (group, pid), g in O.groupby(["group", "photo"]):
    best = g.loc[g.inliers.idxmax()]
    std = g[(g.heading.isin([-5.0, 0.0, 5.0])) & (g.scale.isin([1.0, 1.05]))]
    summary.append(dict(group=group, photo=pid, height=round(best.height), best_inliers=best.inliers, best_heading=best.heading, best_scale=best.scale,
                        best_shift_sigma=round(best.shift_sigma, 2) if best.inliers > 0 else np.nan, std_grid_best=std.inliers.max(), at_0_1=float(g[(g.heading == 0) & (g.scale == 1.0)].inliers.iloc[0]),
                        cells_ge8=int((g.inliers >= 8).sum()), cells=len(g)))
SUM = pd.DataFrame(summary); SUM.to_csv(T7 / "idea0_oracle_summary.csv", index=False)
print(SUM.groupby("group")[["best_inliers", "std_grid_best", "at_0_1", "cells_ge8"]].describe().T.round(1).to_string())
SUM

group                control  skipped
best_inliers  count      8.0     12.0
              mean      21.4      4.4
              std       19.5      3.6
              min        7.0      0.0
              25%        8.5      3.0
              50%       13.5      4.0
              75%       25.0      5.0
              max       63.0     14.0
std_grid_best count      8.0     12.0
              mean      17.6      1.8
              std       19.6      2.6
              min        2.0      0.0
              25%        7.0      0.0
              50%       10.5      0.0
              75%       18.8      3.5
              max       63.0      7.0
at_0_1        count      8.0     12.0
              mean      11.9      0.0
              std       13.2      0.0
              min        0.0      0.0
              25%        3.0      0.0
              50%        8.5      0.0
              75%       16.5      0.0
              max       40.0      0.0
cells_ge8     count      8.0     12.0
            

,group,photo,height,best_inliers,best_heading,best_scale,best_shift_sigma,std_grid_best,at_0_1,cells_ge8,cells
0,control,1,126,63.0,5.0,1.00,0.40,63.0,40.0,21,49
1,control,4,100,7.0,0.0,1.05,0.23,7.0,0.0,0,49
2,control,6,100,9.0,0.0,0.95,0.75,7.0,7.0,3,49
3,control,8,150,37.0,-10.0,1.05,0.62,24.0,18.0,24,49
4,control,31,120,15.0,15.0,0.95,0.20,9.0,4.0,10,49
5,control,36,100,7.0,-10.0,0.95,1.14,2.0,0.0,0,49
6,control,45,139,21.0,-5.0,0.95,0.20,17.0,16.0,18,49
7,control,55,150,12.0,0.0,1.10,0.49,12.0,10.0,10,49
8,skipped,11,108,0.0,-15.0,0.80,NaN,0.0,0.0,0,49
9,skipped,35,120,0.0,-15.0,0.80,NaN,0.0,0.0,0,49
